In [8]:
import re
from pathlib import Path

In [9]:
DOCS_DIR_RAW   = 'primevue/component-documentation/raw'
DOCS_DIR_CLEANED = 'primevue/component-documentation/cleaned'

INPUT_DIR  = Path(DOCS_DIR_RAW)
OUTPUT_DIR = Path(DOCS_DIR_CLEANED)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_files = sorted(INPUT_DIR.glob('*.md'))
print(f'Input-File: {len(input_files)}')
for f in input_files:
    print(f'  {f.name}')

Input-File: 25
  accordion.md
  avatar.md
  badge.md
  breadcrumb.md
  button.md
  card.md
  checkbox.md
  datatable.md
  datepicker.md
  dialog.md
  divider.md
  inputnumber.md
  inputtext.md
  menu.md
  password.md
  popover.md
  progressbar.md
  radiobutton.md
  select.md
  skeleton.md
  slider.md
  tabs.md
  tag.md
  textarea.md
  toggleswitch.md


In [10]:
def extract_import(content: str) -> str:
    """Extracts the ## Import section containing the JavaScript code block."""
    m = re.search(
        r'(## Import\s+```javascript\s+.+?```)',
        content, re.DOTALL
    )

    return m.group(1).strip() if m else ''


def extract_basic(content: str) -> str:
    """Extracts the ## Basic section.

    Fallback: the first section after ## Accessibility,
    if ## Basic is not present.
    """
    m = re.search(
        r'(## Basic\s+.+?)(?=\n## |\Z)',
        content, re.DOTALL
    )

    if not m:
        m = re.search(
            r'## Accessibility.*?\n(## (?!Accessibility).+?)(?=\n## |\Z)',
            content, re.DOTALL
        )

    if not m:
        return ''

    raw = m.group(1)
    raw = re.sub(r'<details>.*?</details>', '', raw, flags=re.DOTALL)
    raw = re.sub(r'\n{3,}', '\n\n', raw)

    return raw.strip()


def extract_template(content: str) -> str:
    """Extracts the ## Template or ## Date Template section.

    Both variants are normalized to the ## Template header.
    """
    m = re.search(
        r'(## (?:Date )?Template\s+.+?)(?=\n## |\Z)',
        content, re.DOTALL
    )

    if not m:
        return ''

    raw = m.group(1)
    raw = re.sub(r'<details>.*?</details>', '', raw, flags=re.DOTALL)
    raw = re.sub(r'^## Date Template', '## Template', raw, flags=re.MULTILINE)
    raw = re.sub(r'\n{3,}', '\n\n', raw)

    return raw.strip()


def extract_advanced(content: str) -> str:
    """Extracts the ## Advanced section.

    Returns an empty string if no Advanced section is present.
    """
    m = re.search(
        r'(## Advanced\s+.+?)(?=\n## |\Z)',
        content, re.DOTALL
    )

    if not m:
        return ''

    raw = m.group(1)
    raw = re.sub(r'<details>.*?</details>', '', raw, flags=re.DOTALL)
    raw = re.sub(r'\n{3,}', '\n\n', raw)

    return raw.strip()


def extract_props(content: str) -> str:
    """Extracts the props section (## ComponentName → ### Props → table).

    Returns an empty string if there is no Props section.
    """
    m = re.search(
        r'(## [A-Z][A-Za-z\s]+\n+### Props\n+\|.+?)(?=\n## |\Z)',
        content, re.DOTALL
    )

    return m.group(1).strip() if m else ''

In [11]:
def build_cleaned_doc(filename: str, content: str) -> str:
    """Assemble the formatted Markdown document from the three sections."""
    component_name = Path(filename).stem.replace('-', ' ').title()

    import_section   = extract_import(content)
    basic_section    = extract_basic(content)
    template_section = extract_template(content)
    advanced_section = extract_advanced(content)
    props_section    = extract_props(content)

    parts = [f'# {component_name}']

    if import_section:   parts.append(import_section)
    if basic_section:    parts.append(basic_section)
    if template_section: parts.append(template_section)
    if advanced_section: parts.append(advanced_section)
    if props_section:    parts.append(props_section)

    return '\n\n'.join(parts) + '\n'

In [12]:
results = []

for path in input_files:
    content = path.read_text(encoding='utf-8', errors='ignore')
    cleaned = build_cleaned_doc(path.name, content)

    out_path = OUTPUT_DIR / path.name
    out_path.write_text(cleaned, encoding='utf-8')

    has_import = '## Import'         in cleaned
    has_basic  = any(f'## {s}' in cleaned for s in [
        'Basic','AvatarGroup','Buttons','Disabled','Card'
    ])
    has_template = '## Template' in cleaned
    has_advanced = '## Advanced' in cleaned
    has_props  = '### Props'          in cleaned

    size_before = len(content)
    size_after  = len(cleaned)
    reduction   = 100 * (1 - size_after / size_before)

    results.append({
        'file':       path.name,
        'has_import': has_import,
        'has_basic':  has_basic,
         'has_template': has_template,
        'has_advanced': has_advanced,
        'has_props':  has_props,
        'bytes_before': size_before,
        'bytes_after':  size_after,
        'reduction_pct': round(reduction, 1),
    })

    warn_status = ''
    if not has_import: warn_status += ' [no import]'
    if not has_basic:  warn_status += ' [no basic]'
    if not has_props:  warn_status += ' [no props]'

    info_tags = ''
    if has_template: info_tags += ' [+template]'
    if has_advanced: info_tags += ' [+advanced]'

    print(f"  {'OK  ' if not warn_status else 'WARN'} "
          f"{path.name:25s} "
          f"{size_before:6d} → {size_after:5d} bytes "
          f"({reduction:5.1f}% smaller)"
          f"{warn_status}{info_tags}")

  OK   accordion.md               32365 →  7245 bytes ( 77.6% smaller) [+template]
  OK   avatar.md                  10949 →  1729 bytes ( 84.2% smaller)
  OK   badge.md                    7442 →  1135 bytes ( 84.7% smaller)
  WARN breadcrumb.md               3953 →   677 bytes ( 82.9% smaller) [no props] [+template]
  OK   button.md                  58691 → 15283 bytes ( 74.0% smaller) [+template]
  OK   card.md                     5705 →  2005 bytes ( 64.9% smaller) [+advanced]
  OK   checkbox.md                17657 →  2507 bytes ( 85.8% smaller)
  OK   datatable.md               82659 →  9920 bytes ( 88.0% smaller) [+template]
  OK   datepicker.md              47012 →  7020 bytes ( 85.1% smaller) [+template]
  OK   dialog.md                  38670 →  5397 bytes ( 86.0% smaller) [+template]
  OK   divider.md                 19508 →  2902 bytes ( 85.1% smaller)
  OK   inputnumber.md             28045 →  6536 bytes ( 76.7% smaller)
  OK   inputtext.md               26659 → 14599 bytes

In [13]:
total_before  = sum(r['bytes_before'] for r in results)
total_after   = sum(r['bytes_after']  for r in results)
avg_reduction = 100 * (1 - total_after / total_before)

ok_count       = sum(1 for r in results if r['has_import'] and r['has_basic'] and r['has_props'])
warn_count     = len(results) - ok_count
template_count = sum(1 for r in results if r['has_template'])
advanced_count = sum(1 for r in results if r['has_advanced'])

print(f'Files processed:                    {len(results)}')
print(f'Complete (Import + Basic + Props):  {ok_count}')
print(f'Incomplete:                         {warn_count}')
print(f'With Template section:              {template_count}')
print(f'With Advanced section:              {advanced_count}')
print(f'Previous size:                      {total_before:,} Bytes')
print(f'Size afterwards:                    {total_after:,} Bytes')
print(f'Average reduction:                  {avg_reduction:.1f}%')
print(f'~Tokens before:                     {total_before // 4:,}')
print(f'~Tokens after:                      {total_after  // 4:,}')

if warn_count:
    print('\nFiles without a complete section:')
    for r in results:
        missing = []
        if not r['has_import']: missing.append('Import')
        if not r['has_basic']:  missing.append('Basic')
        if not r['has_props']:  missing.append('Props')
        if missing:
            print(f"  {r['file']:25s}  missing: {', '.join(missing)}")

Files processed:                    25
Complete (Import + Basic + Props):  23
Incomplete:                         2
With Template section:              13
With Advanced section:              1
Previous size:                      603,083 Bytes
Size afterwards:                    140,052 Bytes
Average reduction:                  76.8%
~Tokens before:                     150,770
~Tokens after:                      35,013

Files without a complete section:
  breadcrumb.md              missing: Props
  menu.md                    missing: Props
